# Plot Friction Tensor

This notebook can be used to to visualize and analyze the elements of the $12 \times 12$ friction tensor provided by the Fortran program for 4 solute particles.

It allows plotting both the diagonal elements and the off-diagonal elements (cross-coupling).

*Note on Column Indexing*: Since the tensor is $12 \times 12$, the data file contains 144 columns after the time column (index 0). The index for element $(i, j)$ is given by `1 + (i * 12) + j` (where $i$ and $j$ go from $0$ to $11$). 
For example, the $\xi_{ZZ}$ component of the first solute corresponds to $i=2, j=2$, which is column `27`.

## Import Libraries

To use this notebook, the following libraries need to be imported:
* `Numpy`
* `Matplotlib`

Make sure that the output files from your simulation, namely `friction_tensor.dat`, `equilibration_stats.dat` and `energy.dat`, are located in the same directory as this notebook before running the cells below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Plot of potential energy during Equilibration

To check whether the number of steps in the initial equilibration phase is sufficient, one can examine the time evolution of the potential energy and the mean squared displacement during this phase. Run the cell below to visualize these plots.

In [ ]:
data = np.genfromtxt("equilibration_stats.dat", comments="#")
time = data[:, 0]
epot = data[:, 1]

plt.figure(figsize=(8,4))
plt.plot(time, epot, label="Potential Energy", color='tab:blue')
plt.xlabel("Time / ps")
plt.ylabel(r"$E_\text{pot}$")
plt.title("Equilibration Phase")
plt.legend()
plt.tight_layout()
plt.show()

## Verifying Average Temperature During Production Phase

In an NVE ensemble, the total energy is conserved, but it continuously partitions between kinetic and potential energy. Because the instantaneous temperature is directly proportional to the kinetic energy, it will naturally fluctuate.

In [ ]:
# Simulation parameters
n_solv = 263
k_b = 0.831

N_df = 3 * n_solv - 3

data = np.genfromtxt("energy.dat", comments="#")
time = data[:, 0]
e_kin = data[:, 1]

# Compute instantaneous temperature array
T_inst = 2.0 * e_kin / (N_df * k_b)

# Compute mean temperature and variance
T_mean = np.mean(T_inst)
T_var = np.var(T_inst, ddof=1) # using ddof=1 for sample variance

print(f"Target Average Temperature: {T_mean:.2f} K")

# Calculate Cv using the fluctuation formula
# (sigma_T^2 / <T>^2) = (2 / N_df) - (k_B / C_v)
fluctuation_ratio = T_var / (T_mean**2)
inverse_term = (2.0 / N_df) - fluctuation_ratio

if inverse_term > 0:
    C_v = k_b / inverse_term
    cv_per_particle = C_v / n_solv
    print(f"Estimated C_v: {C_v:.4f}")
    print(f"Estimated C_v / N_solv: {cv_per_particle:.4f}")
else:
    print("Error: Fluctuation term is larger than 2/N_df")

In [ ]:
column_names = ['Time', 'E_Kin', 'E_Pot', 'E_Tot']
df = pd.read_csv('energy.dat', sep='\s+', comment='#', header=None, names=column_names)

plt.figure(figsize=(8,4))
plt.plot(df['Time'], df['E_Kin'], label='Kinetic Energy', linewidth=1.5)
plt.plot(df['Time'], df['E_Pot'], label='Potential Energy', linewidth=1.5)
plt.plot(df['Time'], df['E_Tot'], label='Total Energy', linewidth=2.0, color='black')
plt.xlabel('Time (ps)', fontsize=14)
plt.ylabel('Energy (a.u.)', fontsize=14)
plt.legend(loc='lower center', bbox_to_anchor=(0.5, 0.4), ncol=3, fontsize=12)
plt.tick_params(axis='both', which='major', labelsize=12)
plt.tight_layout()
plt.show()

## Plot of the 12 diagonal elements of the friction tensor

Run the cell below to display the four plots showing the three diagonal elements of the friction tensor as a function of time, for each of the four solute particles.

In [ ]:
raw = np.genfromtxt("friction_tensor.dat", comments="#", dtype=float)

time = raw[:,0].astype(float)

n_solutes = 4
components = ['X', 'Y', 'Z']
colors = ['r', 'g', 'b']

In [ ]:
fig, axes = plt.subplots(n_solutes, 1, figsize=(8, 14), sharex=True)

if n_solutes == 1:
    axes = [axes]

for solute_idx in range(n_solutes):
    ax = axes[solute_idx]

    for c_idx, comp in enumerate(components):

        i = (solute_idx * 3) + c_idx
        
        col_idx = 1 + (i * 12) + i
        
        data_col = raw[:, col_idx]
        
        ax.plot(time, data_col, label=rf'$\xi_{{{comp}{comp}}}$', color=colors[c_idx], linewidth=1.5, alpha=0.8)

    ax.set_ylabel(r"Friction / Da ps$^{-2}$", fontsize=10)
    ax.legend(loc='upper right')
    ax.set_title(f'Solute {solute_idx + 1} - Diagonal Elements', fontsize=10, fontweight='bold')

axes[-1].set_xlabel("Time / ps", fontsize=10)
plt.tight_layout()
plt.show()

Run the cell below to display the same diagonal elements, but zoomed into the first few picoseconds. This helps to better visualize the initial decay of the autocorrelation function. The time range can be adjusted by modifying `ax.set_xlim(0, 5)` in the code.

In [ ]:
fig, axes = plt.subplots(n_solutes, 1, figsize=(8, 14), sharex=True)
if n_solutes == 1:
    axes = [axes]

for solute_idx in range(n_solutes):
    ax = axes[solute_idx]
    
    for c_idx, comp in enumerate(components):
        i = (solute_idx * 3) + c_idx
        col_idx = 1 + (i * 12) + i  # Diagonale
        
        ax.plot(time, raw[:, col_idx], label=rf'$\xi_{{{comp}{comp}}}$', color=colors[c_idx], linewidth=1.5)

    ax.set_ylabel(r"Friction / Da ps$^{-2}$", fontsize=10)
    ax.legend(loc='upper right')
    ax.set_title(f'Solute {solute_idx + 1} - Diagonal Elements (zoom 0-3 ps)', fontsize=10, fontweight='bold')
    
    ax.set_xlim(0, 3)

axes[-1].set_xlabel("Time / ps", fontsize=10)
plt.tight_layout()
plt.show()

## Plot of the off-diagonal elements of the friction tensor

Run the cell below to display four subplots showing selected non-diagonal elements of the friction tensor (e.g., the cross-terms $\xi_{XY}$, $\xi_{XZ}$, $\xi_{YZ}$) as a function of time, for each of the four solute particles. You can choose which pairs to plot by modifying the `off_diag_pairs` list in the code.

In [ ]:
fig, axes = plt.subplots(n_solutes, 1, figsize=(8, 14), sharex=True)
if n_solutes == 1:
    axes = [axes]

off_diag_pairs = [(0, 1), (0, 2), (1, 2)]             # scegliere a piacimento gli elementi
pair_labels = ['XY', 'XZ', 'YZ']
pair_colors = ['orange', 'purple', 'brown']

for solute_idx in range(n_solutes):
    ax = axes[solute_idx]
    
    for p_idx, (a, b) in enumerate(off_diag_pairs):
        i = (solute_idx * 3) + a
        j = (solute_idx * 3) + b
        
        col_idx = 1 + (i * 12) + j
        
        data_col = raw[:, col_idx]
        
        ax.plot(time, data_col, label=rf'$\xi_{{{pair_labels[p_idx]}}}$', color=pair_colors[p_idx], linewidth=1.5, alpha=0.8)

    ax.set_ylabel(r"Friction / Da ps$^{-2}$", fontsize=10)
    ax.legend(loc='upper right')
    ax.set_title(f'Solute {solute_idx + 1} - Off-Diagonal Elements', fontsize=10, fontweight='bold')

axes[-1].set_xlabel("Time / ps", fontsize=10)
plt.tight_layout()
plt.show()

Run the cell below to visualize the early-time behavior of these off-diagonal terms. As before, the zoom time range can be adjusted by changing `ax.set_xlim(0, 5)`.

In [ ]:
fig, axes = plt.subplots(n_solutes, 1, figsize=(8, 14), sharex=True)
if n_solutes == 1:
    axes = [axes]

for solute_idx in range(n_solutes):
    ax = axes[solute_idx]
    
    for p_idx, (a, b) in enumerate(off_diag_pairs):
        i = (solute_idx * 3) + a
        j = (solute_idx * 3) + b
        col_idx = 1 + (i * 12) + j  # Fuori diagonale
        
        ax.plot(time, raw[:, col_idx], label=rf'$\xi_{{{pair_labels[p_idx]}}}$', color=pair_colors[p_idx], linewidth=1.5)

    ax.set_ylabel(r"Friction / Da ps$^{-2}$", fontsize=10)
    ax.legend(loc='upper right')
    ax.set_title(f'Solute {solute_idx + 1} - Off-Diagonal (zoom 0-5 ps)', fontsize=10, fontweight='bold')
    
    # Imposto lo zoom sull'asse X
    ax.set_xlim(0, 5)

axes[-1].set_xlabel("Time / ps", fontsize=10)
plt.tight_layout()
plt.show()

## 5. Heatmap of the Friction Tensor

Run the cell below to visualize the entire $12 \times 12$ friction tensor as a heatmap at a specific time step. 

The matrix is visually divided into $3 \times 3$ blocks:
* The **diagonal blocks** represent the self-friction of each of the 4 solute particles.
* The **off-diagonal blocks** represent the hydrodynamic cross-coupling between different solutes.

Red colors indicate positive friction values, while blue colors indicate negative values. To view the tensor at a different simulation time, simply change the variable `t_idx` in . $t=0$).

In [ ]:
t_idx = 0
current_time = time[t_idx]

tensor_flat = raw[t_idx, 1:145]
tensor_matrix = tensor_flat.reshape((12, 12))

fig, ax = plt.subplots(figsize=(8, 6))

max_val = np.max(np.abs(tensor_matrix))
cax = ax.imshow(tensor_matrix, cmap='coolwarm', origin='upper', vmin=-max_val, vmax=max_val)

cbar = fig.colorbar(cax)
cbar.set_label(r"Friction / Da ps$^{-2}$", fontsize=10)

ax.set_title(f'12x12 Friction Tensor at t = {current_time:.1f}', fontsize=12, fontweight='bold')

ax.set_xticks(np.arange(-0.5, 12, 3), minor=True)
ax.set_yticks(np.arange(-0.5, 12, 3), minor=True)
ax.grid(which='minor', color='black', linestyle='-', linewidth=1.5)

ax.set_xticks(np.arange(12))
ax.set_yticks(np.arange(12))
ax.set_xticklabels([])
ax.set_yticklabels([])

plt.tight_layout()
plt.show()